In [1]:
import torch
import torch.nn as nn
from torch.cuda.amp import GradScaler, autocast
import timm
import mlflow
from tqdm.notebook import tqdm
import pandas as pd

Download data

In [2]:
import kagglehub

path = kagglehub.dataset_download("tristanzhang32/ai-generated-images-vs-real-images")

print("Path to dataset files:", path)

Utils

In [3]:
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import os

In [4]:
import os
import torch
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from torchvision import transforms
import random
import io
import numpy as np

class AIVsHumanFolderDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir = root_dir
        self.transform = transform
        self.samples = []
        
        for class_name in ['real', 'fake']:
            class_path = os.path.join(root_dir, class_name)
            if not os.path.exists(class_path):
                continue
            
            label = 0 if class_name == 'real' else 1
            
            for img_name in os.listdir(class_path):
                if img_name.lower().endswith(('.jpg', '.jpeg', '.png')):
                    img_path = os.path.join(class_path, img_name)
                    self.samples.append({
                        'path': img_path,
                        'label': label,
                        'class': class_name
                    })
        
        print(f"Загружено {len(self.samples)} изображений из {root_dir}")
        real_count = sum(1 for s in self.samples if s['label'] == 0)
        fake_count = sum(1 for s in self.samples if s['label'] == 1)
        print(f"  Real: {real_count}, Fake: {fake_count}")
    
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        sample = self.samples[idx]
        
        try:
            image = Image.open(sample['path']).convert('RGB')
        except Exception as e:
            print(f"Error loading {sample['path']}: {e}")
            noise = np.random.randint(0, 256, (224, 224, 3), dtype=np.uint8)
            image = Image.fromarray(noise)
        
        label = sample['label']
        
        if self.transform:
            image = self.transform(image)
        
        return image, label

class JPEGCompression:
    def __init__(self, quality_range=(50, 90), p=0.7):
        self.quality_range = quality_range
        self.p = p
        
    def __call__(self, img):
        if random.random() > self.p:
            return img
        quality = random.randint(*self.quality_range)
        buffer = io.BytesIO()
        img.save(buffer, format='JPEG', quality=quality)
        buffer.seek(0)
        return Image.open(buffer).copy()

class GaussianNoise:
    def __init__(self, std_range=(5, 20), p=0.3):
        self.std_range = std_range
        self.p = p
        
    def __call__(self, img):
        if random.random() > self.p:
            return img
        img_array = np.array(img)
        std = random.uniform(*self.std_range)
        noise = np.random.normal(0, std, img_array.shape).astype(np.uint8)
        noisy_array = np.clip(img_array.astype(np.int16) + noise, 0, 255).astype(np.uint8)
        return Image.fromarray(noisy_array)

class RandomBlur:
    def __init__(self, kernel_range=(3, 7), p=0.4):
        self.kernel_range = kernel_range
        self.p = p
        
    def __call__(self, img):
        if random.random() > self.p:
            return img
        from torchvision.transforms import functional as F
        kernel_size = random.randrange(self.kernel_range[0], self.kernel_range[1] + 1, 2)
        return F.gaussian_blur(img, kernel_size=[kernel_size, kernel_size])

def get_train_transforms(img_size=224):
    return transforms.Compose([
        transforms.RandomResizedCrop(img_size, scale=(0.7, 1.0)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomRotation(degrees=15),
        transforms.RandAugment(num_ops=2, magnitude=9),
        transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.1),
        JPEGCompression(quality_range=(60, 95), p=0.6),
        RandomBlur(kernel_range=(3, 5), p=0.3),
        GaussianNoise(std_range=(5, 15), p=0.3),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        transforms.RandomErasing(p=0.25, scale=(0.02, 0.15), ratio=(0.3, 3.3), value=0, inplace=False)
    ])

def get_val_transforms(img_size=224):
    return transforms.Compose([
        transforms.Resize(int(img_size * 1.14)),
        transforms.CenterCrop(img_size),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

def get_robustness_transforms(img_size=224):
    return transforms.Compose([
        transforms.Resize(int(img_size * 1.14)),
        transforms.CenterCrop(img_size),
        JPEGCompression(quality_range=(50, 70), p=1.0),
        RandomBlur(kernel_range=(3, 6), p=0.4),
        GaussianNoise(std_range=(10, 20), p=0.4),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

def create_dataloaders(data_root, img_size=224, batch_size=64, num_workers=4, preload_factor=2):
    train_dir = os.path.join(data_root, 'train')
    test_dir = os.path.join(data_root, 'test')
    
    train_dataset = AIVsHumanFolderDataset(train_dir, transform=get_train_transforms(img_size))
    
    from torch.utils.data import random_split
    val_size = int(0.2 * len(train_dataset))
    train_size = len(train_dataset) - val_size
    train_dataset, val_dataset = random_split(
        train_dataset, 
        [train_size, val_size],
        generator=torch.Generator().manual_seed(42)
    )
    val_dataset.dataset.transform = get_val_transforms(img_size)
    
    test_dataset_clean = AIVsHumanFolderDataset(test_dir, transform=get_val_transforms(img_size))
    test_dataset_robust = AIVsHumanFolderDataset(test_dir, transform=get_robustness_transforms(img_size))
    
    print(f"\nРазмеры выборок:")
    print(f"  Train: {len(train_dataset)}")
    print(f"  Val: {len(val_dataset)}")
    print(f"  Test (clean): {len(test_dataset_clean)}")
    print(f"  Test (robust): {len(test_dataset_robust)}")
    
    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=num_workers,
        pin_memory=True,
        drop_last=True,
        prefetch_factor=preload_factor
    )
    
    val_loader = DataLoader(
        val_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=True,
        prefetch_factor=preload_factor
    )
    
    test_loader_clean = DataLoader(
        test_dataset_clean,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=True,
        prefetch_factor=preload_factor
    )
    
    test_loader_robust = DataLoader(
        test_dataset_robust,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=True,
        prefetch_factor=preload_factor
    )
    
    return train_loader, val_loader, test_loader_clean, test_loader_robust

In [5]:
import math

class Trainer:
    def __init__(self, model, train_loader, val_loader, device, model_name=None, lr=3e-4, use_warmup=False, warmup_epochs=1, total_epochs=6, cammulative_grad=None):
        self.device = device
        self.train_loader = train_loader
        self.val_loader = val_loader
        
        self.model = model.to(device)

        self.optimizer = torch.optim.AdamW(
            self.model.parameters(), 
            lr=lr, 
            weight_decay=0.05
        )

        self.cammulative_grad = cammulative_grad
        print(cammulative_grad)
            
        self.criterion = nn.BCEWithLogitsLoss()
        self.scaler = GradScaler()

        self.use_warmup = use_warmup
        self.warmup_epochs = warmup_epochs
        self.total_epochs = total_epochs
        self.current_epoch = 0

        if use_warmup:
            self.scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
                self.optimizer, T_max=total_epochs
            )
            
        
        self.model_name = model_name

    def train_epoch(self, epoch):
        self.model.train()
        total_loss = 0
        index = 0
        
        for images, labels in tqdm(self.train_loader, desc=f"Training {self.model_name}"):
            index += 1
            images, labels = images.to(self.device), labels.to(self.device).float()
            
            with autocast():
                logits = self.model(images).squeeze()
                loss = self.criterion(logits, labels)
                
            self.scaler.scale(loss).backward()

            if index >= self.cammulative_grad:
                self.scaler.step(self.optimizer)
                self.scaler.update()
                self.optimizer.zero_grad()
                index = 0
            total_loss += loss.item()

        if index != 0:
            self.scaler.step(self.optimizer)
            self.scaler.update()
            self.optimizer.zero_grad()

        if not self.use_warmup:
            self.scheduler.step()
            
        return total_loss / len(self.train_loader)

    @torch.no_grad()
    def validate(self):
        self.model.eval()
        all_logits, all_labels = [], []
        
        for images, labels in self.val_loader:
            images = images.to(self.device)
            logits = self.model(images).squeeze()
            
            all_logits.append(logits.cpu())
            all_labels.append(labels)
            
        return torch.cat(all_logits), torch.cat(all_labels)

In [6]:
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, average_precision_score
from scipy.optimize import minimize_scalar
import torch

class Evaluator:
    def __init__(self, logits: torch.Tensor, labels: torch.Tensor):
        self.probs = torch.sigmoid(logits).numpy()
        self.labels = labels.numpy()
        self.logits = logits.numpy()
        
    def calculate_metrics(self, threshold=0.5):
        preds = (self.probs >= threshold).astype(int)
        return {
            "accuracy": accuracy_score(self.labels, preds),
            "precision": precision_score(self.labels, preds),
            "recall": recall_score(self.labels, preds),
            "f1": f1_score(self.labels, preds),
            "roc_auc": roc_auc_score(self.labels, self.probs),
            "average_precision_score": average_precision_score(self.labels, self.probs)
        }

    def temperature_scaling(self):
        def nll_loss(T):
            scaled_probs = 1 / (1 + np.exp(-self.logits / T))
            eps = 1e-15
            scaled_probs = np.clip(scaled_probs, eps, 1 - eps)
            loss = -np.mean(self.labels * np.log(scaled_probs) + (1 - self.labels) * np.log(1 - scaled_probs))
            return loss

        res = minimize_scalar(nll_loss, bounds=(0.1, 10.0), method='bounded')
        optimal_T = res.x
        
        calibrated_probs = 1 / (1 + np.exp(-self.logits / optimal_T))
        return optimal_T, calibrated_probs

    def calculate_ece(self, probs, n_bins=10):
        bin_boundaries = np.linspace(0, 1, n_bins + 1)
        ece = 0.0
        for i in range(n_bins):
            in_bin = (probs > bin_boundaries[i]) & (probs <= bin_boundaries[i+1])
            prop_in_bin = in_bin.mean()
            
            if prop_in_bin > 0:
                avg_confidence = probs[in_bin].mean()
                avg_accuracy = self.labels[in_bin].mean()
                ece += np.abs(avg_accuracy - avg_confidence) * prop_in_bin
                
        return ece

In [7]:
def get_model_size_mb(model):
    param_size = sum(p.nelement() * p.element_size() for p in model.parameters())
    buffer_size = sum(b.nelement() * b.element_size() for b in model.buffers())
    return (param_size + buffer_size) / 1024 / 1024

In [8]:
import torch
import timm
import onnx
import os

def export_to_onnx(model, checkpoint_path:str, onnx_path: str, img_size: int = 224):
    """
    Экспортирует PyTorch модель в формат ONNX.
    """
    model = model.to('cpu')
    model.eval()
    for param in model.parameters():
        param.requires_grad = False
        
    dummy_input = torch.randn(1, 3, img_size, img_size, requires_grad=False)
    
    print(f"[*] Экспорт в ONNX (размер входа: {img_size}x{img_size})...")
    
    torch.onnx.export(
        model,
        dummy_input,
        onnx_path,
        export_params=True,
        opset_version=13,
        do_constant_folding=True,
        input_names=['input'],
        output_names=['output'],
    )
    
    print("[*] Проверка целостности ONNX модели...")
    onnx_model = onnx.load(onnx_path)
    onnx.checker.check_model(onnx_model)
    print(f"Модель успешно экспортирована и сохранена в {onnx_path}")
    
    pt_size = os.path.getsize(checkpoint_path) / (1024 * 1024)
    onnx_size = os.path.getsize(onnx_path) / (1024 * 1024)
    print(f"Размер PyTorch весов: {pt_size:.2f} MB")
    print(f"Размер ONNX весов: {onnx_size:.2f} MB")

In [9]:
import torch
import onnxruntime as ort
import numpy as np
import time
import timm

class ModelBenchmark:
    def __init__(self, pt_model, onnx_path, device="cpu", img_size=224):
        self.device = device
        self.img_size = img_size
        
        self.pt_model = pt_model.to(device)
        self.pt_model.eval()
        
        sess_options = ort.SessionOptions()
        sess_options.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL
        self.ort_session = ort.InferenceSession(onnx_path, sess_options, providers=['CPUExecutionProvider'])
        
    def _get_dummy_input(self, batch_size=1):
        return torch.randn(batch_size, 3, self.img_size, self.img_size)

    def _benchmark_pytorch(self, n_runs=100, warmup=10):
        dummy_input = self._get_dummy_input().to(self.device)
        
        with torch.no_grad():
            for _ in range(warmup):
                _ = self.pt_model(dummy_input)
                
        if self.device == "cuda":
            torch.cuda.synchronize()
            
        times = []
        with torch.no_grad():
            for _ in range(n_runs):
                start = time.perf_counter()
                _ = self.pt_model(dummy_input)
                if self.device == "cuda":
                    torch.cuda.synchronize()
                end = time.perf_counter()
                times.append((end - start) * 1000)
                
        return np.mean(times), np.std(times)

    def _benchmark_onnx(self, n_runs=100, warmup=10):
        dummy_input = self._get_dummy_input().numpy()
        ort_input = {self.ort_session.get_inputs()[0].name: dummy_input}
        
        for _ in range(warmup):
            _ = self.ort_session.run(None, ort_input)
            
        times = []
        for _ in range(n_runs):
            start = time.perf_counter()
            _ = self.ort_session.run(None, ort_input)
            end = time.perf_counter()
            times.append((end - start) * 1000)
            
        return np.mean(times), np.std(times)

    def verify_quality(self):
        dummy_input = self._get_dummy_input()
        
        with torch.no_grad():
            pt_out = self.pt_model(dummy_input.to(self.device)).cpu().numpy()
            
        ort_input = {self.ort_session.get_inputs()[0].name: dummy_input.numpy()}
        onnx_out = self.ort_session.run(None, ort_input)[0]
        
        max_diff = np.max(np.abs(pt_out - onnx_out))
        mean_diff = np.mean(np.abs(pt_out - onnx_out))
        
        return max_diff, mean_diff, pt_out, onnx_out

    def run_full_benchmark(self, n_runs=100):
        print("="*50)
        print("ЗАПУСК ПОЛНОГО БЕНЧМАРКИНГА")
        print("="*50)
        
        max_diff, mean_diff, pt_out, onnx_out = self.verify_quality()
        print(f"\n[Качество] Max absolute difference: {max_diff:.6f}")
        print(f"[Качество] Mean absolute difference: {mean_diff:.6f}")
        if max_diff < 1e-4:
            print("[+] Качество сохранено (разница в пределах погрешности FP32).")
        else:
            print("[-] ВНИМАНИЕ: Разница в выходах превышает допустимую!")
            
        pt_mean, pt_std = self._benchmark_pytorch(n_runs)
        onnx_mean, onnx_std = self._benchmark_onnx(n_runs)
        
        speedup = pt_mean / onnx_mean
        
        print(f"\n[Скорость] PyTorch Latency: {pt_mean:.2f} ± {pt_std:.2f} ms")
        print(f"[Скорость] ONNX Latency:    {onnx_mean:.2f} ± {onnx_std:.2f} ms")
        print(f"[Скорость] ONNX Speedup:    {speedup:.2f}x")
        print(f"[Скорость] PyTorch FPS:     {1000 / pt_mean:.1f}")
        print(f"[Скорость] ONNX FPS:        {1000 / onnx_mean:.1f}")
        print("="*50)
        
        return {
            "pt_latency_ms": pt_mean,
            "onnx_latency_ms": onnx_mean,
            "speedup_factor": speedup,
            "max_diff": max_diff
        }


2026-06-21 10:03:50.197371464 [W:onnxruntime:Default, device_discovery.cc:283 GetGpuDevices] Failed to detect devices under "/sys/class/drm/card0": device_discovery.cc:93 ReadFileContents Failed to open file: "/sys/class/drm/card0/device/vendor"


Loading data

In [10]:
# train_loader, val_loader, test_loader_clean, test_loader_robust = create_dataloaders(
#     data_root=path,
#     img_size=224,
#     batch_size=100,
#     num_workers=6,
#     preload_factor=3
# )

In [11]:
# def optimizer_split_for_swin(model, lr) -> dict:
#     backbone_params = []
#     head_params = []
#     for name, param in model.named_parameters():
#         if "head" in name:
#             head_params.append(param)
#         else:
#             backbone_params.append(param)

#     return [{'params': backbone_params, 'lr': lr}, {'params': head_params, 'lr': lr * 10}]

In [12]:
# def optimizer_split_for_vit(model, lr) -> dict:
#     backbone_params = []
#     head_params = []
#     for name, param in model.named_parameters():
#         if "head" == name:
#             head_params.append(param)
#         else:
#             backbone_params.append(param)

#     return [{'params': backbone_params, 'lr': lr}, {'params': head_params, 'lr': lr * 10}]

## Train and save all models

In [13]:
configs = [
  {
    "model_name": "Swin S3 Transformer (Small)",
    "timm_name": "swin_s3_small_224.ms_in1k",
    "epochs": 20,
    "lr": 5e-5,
    "use_warmup": True,
    "batch_size": 32,
    "cammulative_grad": 8
  },
  # {
  #   "model_name": "Swin S3 Transformer (Base)",
  #   "timm_name": "swin_s3_base_224.ms_in1k",
  #   "epochs": 20,
  #   "lr": 5e-5,
  #   "use_warmup": True,
  #   "batch_size": 32
  # },
  # {
  #   "model_name": "Swin S3 Transformer (Tiny)",
  #   "timm_name": "swin_s3_tiny_224.ms_in1k",
  #   "epochs": 20,
  #   "lr": 5e-5,
  #   "use_warmup": True,
  #   "batch_size": 64
  # },
  # # {
  #   "model_name": "Resnet50",
  #   "timm_name": "resnet50",
  #   "epochs": 10,
  #   "lr": 3e-4,
  #   "batch_size": 64
  # },
  # {
  #   "model_name": "EfficientNetV2‑S",
  #   "timm_name": "tf_efficientnetv2_s.in1k",
  #   "epochs": 10,
  #   "lr": 3e-4,
  #   "batch_size": 64
  # },
  # {
  #   "model_name": "Vision Transformer (ViT, Base)",
  #   "timm_name": "vit_base_patch16_224",
  #   "epochs": 20,
  #   "lr": 5e-5,
  #   "use_warmup": True,
  #   "batch_size": 64
  # },
  # {
  #   "model_name": "MobileNetV3 (Large)",
  #   "timm_name": "mobilenetv3_large_100",
  #   "epochs": 10,
  #   "lr": 3e-4,
  #   "batch_size": 64
  # }
]

In [14]:
thresholds = [0.75, 0.9, 0.95]

In [15]:
import time

In [16]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

In [17]:
mlflow.set_experiment("AI_vs_Human_Classification")

<Experiment: artifact_location='/home/nukfe/projects/fakephoto_detector/notebooks/mlruns/1', creation_time=1781723427719, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1781723427719, lifecycle_stage='active', name='AI_vs_Human_Classification', tags={}, trace_location=None, workspace='default'>

In [18]:
for model_cfg in configs:
    print(f"\n--- Запуск эксперимента: {model_cfg['model_name']} ---")

    train_loader, val_loader, test_loader_clean, test_loader_robust = create_dataloaders(
        data_root=path,
        img_size=224,
        batch_size=model_cfg["batch_size"],
        num_workers=6,
        preload_factor=3
    )
    
    with mlflow.start_run(run_name=model_cfg['model_name'], log_system_metrics=True):
        mlflow.log_param("trainer", str(model_cfg))
        mlflow.log_param("model_arch", model_cfg['timm_name'])

        model = timm.create_model(model_cfg['timm_name'], pretrained=True, num_classes=1)
        
        trainer = Trainer(model, train_loader, test_loader_robust, device,
                          model_name=model_cfg.get("model_name"),
                          lr=model_cfg.get("lr"),
                          use_warmup=model_cfg.get("use_warmup"),
                          total_epochs=model_cfg.get("epochs"),
                          cammulative_grad=model_cfg.get("cammulative_grad"))
        max_av_prec_score = 0
        
        start_time = time.time()
        for epoch in range(model_cfg['epochs']):
            loss = trainer.train_epoch(epoch)
            mlflow.log_metric("train_loss", loss, step=epoch)

            val_logits, val_labels = trainer.validate()
            evaluator = Evaluator(val_logits, val_labels)

            av_prec_score = evaluator.calculate_metrics()["average_precision_score"]
            if max_av_prec_score < av_prec_score:
                torch.save(trainer.model.state_dict(), f"models/{model_cfg['timm_name']}_best.pth")
                mlflow.log_artifact(f"models/{model_cfg['timm_name']}_best.pth")
                max_av_prec_score = max_av_prec_score
            
            for threshold in thresholds:
                metrics = evaluator.calculate_metrics(threshold)
                for metric, value in metrics.items():
                    mlflow.log_metric(f"{metric}_per_threshold_{threshold}", str(value), step=epoch)
        
        training_time = time.time() - start_time
        mlflow.log_metric("training_time_sec", training_time)
        model.load_state_dict(torch.load(f"models/{model_cfg['timm_name']}_best.pth", map_location=device))
        
        val_logits, val_labels = trainer.validate()
        evaluator = Evaluator(val_logits, val_labels)
        
        optimal_T, calibrated_probs = evaluator.temperature_scaling()
        ece_before = evaluator.calculate_ece(evaluator.probs)
        ece_after = evaluator.calculate_ece(calibrated_probs)
        
        mlflow.log_metric("optimal_temperature", optimal_T)
        mlflow.log_metric("ECE_before_calibration", ece_before)
        mlflow.log_metric("ECE_after_calibration", ece_after)
        
        metrics = evaluator.calculate_metrics()
        for threshold in thresholds:
            metrics = evaluator.calculate_metrics(threshold)
            for metric, value in metrics.items():
                mlflow.log_metric(f"{metric}_per_threshold_{threshold}_final", str(value))
        
        model_size_mb = get_model_size_mb(trainer.model)
        mlflow.log_metric("model_size_mb", model_size_mb)

        export_to_onnx(model, f"models/{model_cfg['timm_name']}_best.pth", f"models/onnx/{model_cfg['timm_name']}.onnx")
        mlflow.log_artifact(f"models/onnx/{model_cfg['timm_name']}.onnx")
        
        print(f"Модель {model_cfg['timm_name']} обучена. F1: {metrics['f1']:.4f}, ECE: {ece_after:.4f}")
        torch.cuda.empty_cache()


--- Запуск эксперимента: Swin S3 Transformer (Small) ---
Загружено 48000 изображений из /home/nukfe/.cache/kagglehub/datasets/tristanzhang32/ai-generated-images-vs-real-images/versions/2/train
  Real: 24000, Fake: 24000
Загружено 12000 изображений из /home/nukfe/.cache/kagglehub/datasets/tristanzhang32/ai-generated-images-vs-real-images/versions/2/test
  Real: 6000, Fake: 6000
Загружено 12000 изображений из /home/nukfe/.cache/kagglehub/datasets/tristanzhang32/ai-generated-images-vs-real-images/versions/2/test
  Real: 6000, Fake: 6000

Размеры выборок:
  Train: 38400
  Val: 9600
  Test (clean): 12000
  Test (robust): 12000


2026/06/21 10:03:52 INFO mlflow.system_metrics.system_metrics_monitor: Skip logging GPU metrics. Set logger level to DEBUG for more details.
2026/06/21 10:03:52 INFO mlflow.system_metrics.system_metrics_monitor: Started monitoring system metrics.


8


/tmp/ipykernel_23901/2785066902.py:21: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = GradScaler()


Training Swin S3 Transformer (Small):   0%|          | 0/1200 [00:00<?, ?it/s]

...

The model version conversion is not supported by the onnxscript version converter and fallback is enabled. The model will be converted using the onnx C API (target version: 13).


[torch.onnx] Translate the graph into ONNX... ✅


Failed to convert the model to the target version 13 using the ONNX C API. The model was not modified
Traceback (most recent call last):
  File "/home/nukfe/projects/fakephoto_detector/.venv/lib/python3.12/site-packages/onnxscript/version_converter/__init__.py", line 137, in call
    converted_proto = _c_api_utils.call_onnx_api(
                      ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/nukfe/projects/fakephoto_detector/.venv/lib/python3.12/site-packages/onnxscript/version_converter/_c_api_utils.py", line 65, in call_onnx_api
    result = func(proto)
             ^^^^^^^^^^^
  File "/home/nukfe/projects/fakephoto_detector/.venv/lib/python3.12/site-packages/onnxscript/version_converter/__init__.py", line 132, in _partial_convert_version
    return onnx.version_converter.convert_version(
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/nukfe/projects/fakephoto_detector/.venv/lib/python3.12/site-packages/onnx/version_converter.py", line 39, in convert_version
    conv

[torch.onnx] Optimize the ONNX graph...
[torch.onnx] Optimize the ONNX graph... ✅
[*] Проверка целостности ONNX модели...
Модель успешно экспортирована и сохранена в models/onnx/swin_s3_small_224.ms_in1k.onnx
Размер PyTorch весов: 186.93 MB
Размер ONNX весов: 2.66 MB
Модель swin_s3_small_224.ms_in1k обучена. F1: 0.6786, ECE: 0.1092


2026/06/21 12:52:44 INFO mlflow.system_metrics.system_metrics_monitor: Stopping system metrics monitoring...
2026/06/21 12:52:44 INFO mlflow.system_metrics.system_metrics_monitor: Successfully terminated system metrics monitoring!


In [17]:
mlflow.end_run()